# Silage or Grain? Field-level corn harvest classification (Maryland, USDA BARC)

**Question (USDA NASS):** for each corn field, was it harvested as **silage** (chopped green, late Aug to Sep) or **grain** (dried down, combined Oct to Nov), and can we say so **before the January Annual Summary**, in a way a skeptic can check?

**Why it matters:** production = harvested acres x yield. Silage acres come straight out of grain acres, so an earlier silage estimate sharpens the corn production forecast.

**Approach in one line:** per-field satellite index time series -> transparent rules (baseline) and a small interpretable model -> per-field class with evidence, an abstain option, and an acre roll-up with an uncertainty interval checked against NASS.

## 1. Setup
All paths come from environment variables so the same notebook runs on Keeboh or a laptop.

In [ ]:
import os, json, sys
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "Submission" else Path.cwd()
sys.path.insert(0, str(REPO))
os.environ.setdefault("HANDOFF_DIR", "/workspace/data/handoff")
OUT = Path(os.environ.get("RUN_OUT", "/workspace/data/submission_run"))
YEARS = [2021, 2022, 2023]
AS_OF = "10-12"          # month-day used for the headline numbers
SOURCE = "auto"          # ML predictions if delivered, else rule baseline

import pandas as pd, matplotlib.pyplot as plt
from harvest import io, tools, baseline
from harvest.run import run
print("hand-off folder:", io.handoff_dir())
io.status()

## 2. Data
- **Fields:** USDA BARC (Beltsville, MD) field polygons, keyed by `OBJECTID`.
- **Labels:** 44 field-years (32 grain, 12 silage), 2021 to 2023.
- **Imagery:** Sentinel-2 and Landsat surface reflectance, cloud-masked, averaged per field (EPSG:32618 for area).
- **Indices:** NDVI, EVI, NDRE (greenness), NDTI (crop residue), clear-sky share per date.
- **NASS QuickStats:** Maryland corn silage and grain acres, 2012 to 2025.

In [ ]:
fields = io.read_table("fields"); labels = io.read_table("labels"); fd = io.read_table("field_dates")
print(len(fields), "fields |", len(labels), "labeled field-years |", len(fd), "field-date rows")
labels.groupby(["year", "label"]).size().unstack(fill_value=0)

## 3. What silage and grain look like
Silage: green right up to a sudden drop to bare soil in late summer. Grain: weeks of gradual browning, then stalk residue (higher NDTI) after harvest.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, lab in zip(axes, ["Silage", "Grain"]):
    for _, r in labels[labels.label == lab].head(4).iterrows():
        g = fd[(fd.OBJECTID == r.OBJECTID) & (fd.year == r.year)].sort_values("date")
        ax.plot(pd.to_datetime(g.date).dt.dayofyear, g.ndvi, marker="o", ms=3, alpha=.8, label=f"{r.OBJECTID}/{r.year}")
    ax.set_title(lab); ax.set_xlabel("day of year"); ax.legend(fontsize=7)
axes[0].set_ylabel("NDVI"); plt.tight_layout()

## 4. Rule baseline (readable by a skeptic)
Using only clear looks up to the as-of date:
1. Fewer than 4 clear looks -> **Insufficient** (abstain).
2. Biggest NDVI drop green (>= 0.60) to bare (<= 0.35), before ~Sep 25, little residue after -> **Silage**.
3. Sharp drop that fails a silage test, or slow brown-down -> **Grain**.
4. Still green -> **Pending**.

Thresholds are named constants in `harvest/baseline.py`.

In [ ]:
{k: getattr(baseline, k) for k in ["MIN_CLEAR", "MIN_LOOKS", "GREEN", "BARE", "DROP", "SILAGE_BY", "RESIDUE"]}

## 5. Results by held-out year
Each year is scored on its own, so no year's labels leak into another. We report **balanced accuracy** and **silage F1** next to the 'always grain' score, because 'always grain' wins on plain accuracy.

In [ ]:
rows, results = [], {}
for y in YEARS:
    res = run(y, f"{y}-{AS_OF}", OUT / str(y), SOURCE); results[y] = res
    m = json.loads((OUT / str(y) / "metrics.json").read_text())
    rows.append({"year": y, **{k: m[k] for k in ["n_labeled", "n_decided", "abstain_rate", "accuracy",
                 "balanced_accuracy", "silage_precision", "silage_recall", "silage_f1",
                 "always_grain_accuracy"]}, "model": ",".join(m["models"])})
metrics = pd.DataFrame(rows).set_index("year"); metrics

In [ ]:
fig, axes = plt.subplots(1, len(YEARS), figsize=(4 * len(YEARS), 3))
for ax, y in zip(axes, YEARS):
    cm = json.loads((OUT / str(y) / "metrics.json").read_text())["confusion_matrix"]
    ax.imshow(cm["values"], cmap="Greens")
    ax.set_xticks(range(len(cm["cols_pred"])), cm["cols_pred"], rotation=30); ax.set_yticks(range(2), cm["rows_true"])
    for i, row in enumerate(cm["values"]):
        for j, v in enumerate(row): ax.text(j, i, v, ha="center", va="center")
    ax.set_title(f"{y} as of {AS_OF}"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout()

## 6. How early can we call it?
Class counts and silage F1 as the as-of date moves through the season. Every call here is months before the January Annual Summary.

In [ ]:
fig, axes = plt.subplots(1, len(YEARS), figsize=(4 * len(YEARS), 3), sharey=True)
for ax, y in zip(axes, YEARS):
    sw = pd.DataFrame(json.loads((OUT / str(y) / "asof_sweep.json").read_text()))
    counts = pd.DataFrame(list(sw["counts"])).fillna(0); counts.index = pd.to_datetime(sw["as_of"])
    counts.plot.area(ax=ax, alpha=.7, legend=(y == YEARS[0])); ax.set_title(str(y)); ax.set_xlabel("as of")
axes[0].set_ylabel("fields"); plt.tight_layout()

## 7. Roll-up to acres and NASS sanity check
Expected silage acres = sum of p(silage) x acres; the 95% interval treats fields as independent. BARC is a small sample of Maryland, so we compare the **silage share** with the state share rather than raw acres.

In [ ]:
roll = []
for y in YEARS:
    r = json.loads((OUT / str(y) / "rollup.json").read_text())
    roll.append({"year": y, "silage_acres_expected": r["silage_acres_expected"], "ci95": r["silage_acres_95ci"],
                 "silage_share_of_decided": r["silage_share_of_decided"], "undecided_acres": r["undecided_acres"],
                 "nass_rows": len(r["nass"] or [])})
pd.DataFrame(roll).set_index("year")

In [ ]:
try:
    qs = pd.DataFrame(tools.get_quickstats("MARYLAND"))
    display(qs.head(20))
except FileNotFoundError as e:
    print("QuickStats CSV not found:", e)

## 8. Agent trace (Nemotron)
The agent (Analyst -> Verifier -> Writer) calls the same tool functions used above. Each step is logged to `trace.jsonl`.

In [ ]:
p = io.handoff_dir() / "trace.jsonl"
if p.exists():
    tr = pd.DataFrame([json.loads(l) for l in p.read_text().splitlines() if l.strip()])
    display(tr.tail(20))
else:
    print("no trace.jsonl yet")

## 9. Limitations
- 44 labeled field-years from one research farm; per-year silage counts are small, so metrics have wide uncertainty.
- Cloud gaps can hide the cut week; we abstain rather than guess.
- Early harvest is not always silage (e.g., Texas summer grain); rules are tuned for Maryland timing.
- Double crops are not modelled separately.

## 10. Reproduce
```bash
python -m harvest.run --year 2022 --as-of 2022-10-12 --out /workspace/data/run2022
uvicorn api.main:app --host 0.0.0.0 --port 8000
```

## Citations
_TODO: add the required USDA BARC citations here and in the README._